## 2. 웹스크래핑 연습문제

2-1. Nate 뉴스기사 제목 스크래핑하기 (필수) 

In [26]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display

# 섹션별 코드 딕셔너리
section_dict = {
    '최신뉴스': 'n0100', '정치': 'n0200', '경제': 'n0300',
    '사회': 'n0400', '세계': 'n0500', 'IT/과학': 'n0600'
}

def scrape_nate_news(section_name):
    mid = section_dict.get(section_name)
    if not mid:
        print("잘못된 섹션명입니다.")
        return

    base_url = f'https://news.nate.com/recent?mid={mid}'
    
    req_header = {
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
    }
    
    res = requests.get(base_url, headers=req_header)
    soup = BeautifulSoup(res.text, 'html.parser')
    
    print(f"====== [ 네이트 {section_name} 뉴스 ] ======")
    
    articles = soup.select('div.post-item')
    
    for article in articles:
        # [1] 이미지 처리
        img_tag = article.select_one('img')
        
        if img_tag and 'src' in img_tag.attrs:
            raw_src = img_tag['src']
            img_url = urljoin(base_url, raw_src)
            try:
                display(Image(url=img_url, width=120))
            except Exception:
                print("[이미지 로딩 실패]")
        else:
            print("[이미지 없음]")

        # [2] 제목 및 링크 추출 (들여쓰기 수정 완료)
        title_tag = article.select_one('span.tit')
        a_tag = article.select_one('a')
        
        if title_tag and a_tag:
            title = title_tag.text.strip()
            link = urljoin(base_url, a_tag['href'])
            
            print(f"기사제목: {title}")
            print(f"기사링크: {link}")
            
        print("-" * 50)

# 실행
scrape_nate_news('경제')

====== [ 네이트 경제 뉴스 ] ======


2-2. 하나의 네이버 웹툰과 1개의 회차에 대한 Image 다운로드 하기 (필수) 

In [15]:
import os
import requests
from bs4 import BeautifulSoup

def download_one_episode(title, no, url):
    # 1. 디렉토리 생성 (img\제목\회차번호)
    save_path = os.path.join('img', title, str(no))
    os.makedirs(save_path, exist_ok=True)
    
    # 2. 웹툰 페이지 요청 헤더 설정 
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': url
    }

    res = requests.get(url, headers=headers)
    if not res.ok:
        print(f"웹툰 페이지 접속 실패: {res.status_code}")
        return

    soup = BeautifulSoup(res.text, 'html.parser')

    # 3. 웹툰 이미지 태그들 찾기
    img_tags = soup.select("img[src*='IMAG01']")
    
    if not img_tags:
        print("이미지를 찾을 수 없습니다. 선택자를 확인해 주세요.")
        return

    print(f"[{title}] {no}화 다운로드 시작...")

    # 4. 이미지 다운로드 및 저장
    for i, img_tag in enumerate(img_tags):
        img_url = img_tag['src']
        
        # 이미지 데이터 요청
        img_res = requests.get(img_url, headers=headers)
        
        if img_res.ok:
            img_data = img_res.content
            # 파일명 생성 
            file_name = f"{i:03d}_{os.path.basename(img_url.split('?')[0])}"
            file_full_path = os.path.join(save_path, file_name)
            
            # 바이너리 모드로 저장
            with open(file_full_path, 'wb') as f:
                f.write(img_data)
            print(f"저장 완료: {file_name} ({len(img_data):,} bytes)")
        else:
            print(f"이미지 다운로드 실패: {img_url}")

    print(f"\n✅ '{title}' {no}화 모든 이미지 다운로드 완료!")

# 함수 호출 예시
download_one_episode('괴물 천재선수들이 날 너무 좋아함', 9, 'https://comic.naver.com/webtoon/detail?titleId=843901&no=9&week=sun')

[괴물 천재선수들이 날 너무 좋아함] 9화 다운로드 시작...
저장 완료: 000_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_1.jpg (255,530 bytes)
저장 완료: 001_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_2.jpg (203,567 bytes)
저장 완료: 002_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_3.jpg (174,585 bytes)
저장 완료: 003_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_4.jpg (227,371 bytes)
저장 완료: 004_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_5.jpg (274,280 bytes)
저장 완료: 005_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_6.jpg (176,032 bytes)
저장 완료: 006_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_7.jpg (243,124 bytes)
저장 완료: 007_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_8.jpg (161,332 bytes)
저장 완료: 008_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_9.jpg (227,641 bytes)
저장 완료: 009_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_10.jpg (176,666 bytes)
저장 완료: 010_20260106143855_1991d66d8ef4beca0136b7bbe704bfac_IMAG01_11.jpg (146,703 by

2-3. 하나의 네이버 웹툰과 여러개의 회차에 대한 Image 다운로드 하기 (선택)